Imports

In [1]:
import pandas as pd

Read data

In [2]:
data_path = 'D:/GitHub/_data/personal_finances/output/transactions.csv'
df_raw = pd.read_csv(data_path, encoding='utf-8')
df_raw.shape

(100, 19)

Dataprep

In [103]:
df = df_raw.copy()

cols_to_drop = ['payee','user_date','sic','mcc','checknum','number','fid','routing_number']
df.drop(cols_to_drop, axis=1, inplace=True)

df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['year_month'] = df['date'].dt.to_period('M')
df['date'] = df['date'].dt.date

df["amount"] = df["amount"].astype(float)

df['in_out'] = df['amount'].apply(lambda x: 'in' if x > 0 else 'out')

# flag transactions where contains installment number
installment_pattern = r"\b(?:0|[1-9]|[1-9][0-9]|100)/(?:0|[1-9]|[1-9][0-9]|100)\b"
df['is_installment'] = df['memo'].str.contains(installment_pattern)

df.head(2)

,type,date,amount,id,memo,account_id,institution,year,year_month,in_out,is_installment
0,Credit Card,2024-12-04,-38.90,674ea97b-5319-4c30-83bf-47b17323cfe5,Cinemark Brasil,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False
1,Credit Card,2024-12-02,-14.96,674b9f46-f4c2-4ff9-98f9-722110c83e69,Uber Uber *Trip Help.U,5a43cccd-bdaf-4211-a5c5-bc45326df942,NU PAGAMENTOS S.A.,2024,2024-12,out,False


EDA

In [104]:
df.dtypes

type                 object
date                 object
amount              float64
id                   object
memo                 object
account_id           object
institution          object
year                  int32
year_month        period[M]
in_out               object
is_installment         bool
dtype: object

In [105]:
df.isnull().sum()

type              0
date              0
amount            0
id                0
memo              0
account_id        0
institution       0
year              0
year_month        0
in_out            0
is_installment    0
dtype: int64

In [106]:
columns = df.columns
for c in columns:
    display(c)
    display(df[c].value_counts())

'type'

type
Credit Card    73
Bank           27
Name: count, dtype: int64

'date'

date
2024-12-02    8
2024-11-05    8
2024-12-09    6
2024-11-25    6
2024-11-30    6
2024-11-16    6
2024-11-24    5
2024-12-17    5
2024-11-11    4
2024-11-21    4
2024-11-20    3
2024-12-06    3
2024-11-13    3
2024-11-17    3
2024-11-14    3
2024-12-14    2
2024-11-15    2
2024-12-20    2
2024-11-07    2
2024-11-10    2
2024-12-13    2
2024-11-19    2
2024-12-15    2
2024-11-12    2
2024-12-04    1
2024-11-18    1
2024-11-22    1
2024-11-26    1
2024-11-27    1
2024-11-28    1
2024-12-01    1
2024-11-09    1
2024-12-16    1
Name: count, dtype: int64

'amount'

...

year_month
2024-11    67
2024-12    33
Freq: M, Name: count, dtype: int64

'in_out'

in_out
out    87
in     13
Name: count, dtype: int64

'is_installment'

is_installment
False    87
True     13
Name: count, dtype: int64

In [99]:
df.groupby(['type','in_out','year_month'])['amount'].sum().reset_index()

,type,in_out,year_month,amount
0,Bank,in,2024-12,17962.60
1,Bank,out,2024-12,-16295.21
2,Credit Card,in,2024-11,2916.27
3,Credit Card,out,2024-11,-3196.73
4,Credit Card,out,2024-12,-231.14


In [108]:
df['is_installment'].value_counts()
# df.query('is_installment == True')

is_installment
False    87
True     13
Name: count, dtype: int64

Group message to avoid duplicated calls when classify trought LLMs

In [ ]:
#df_raw.query('id == "6733a159-1e07-48ba-bdf3-dc4be7f83570"')
df_grouped = df_raw.groupby('memo')['id'].apply(list).reset_index(name='id_list')
df_grouped['len_id'] = df_grouped['id_list'].apply(lambda x: len(x))
df_grouped

,memo,id_list,len_id
0,1a99,[6737e065-e913-4ef3-85e4-c3b624b73956],1
1,Aplicação RDB,"[674dda0d-b620-4356-a00a-292ae7fbb188, 674dda5...",4
2,Bar e Restaurante do C,[672ffb36-6a6a-4f13-9c48-f3ecb82fe343],1
3,Black Dogs,[67390ddb-d826-4e08-9cf7-24987e328707],1
4,Bns Restaurante,[6746771a-e604-4a25-a8a2-d4a0831acf51],1
...,...,...,...
63,Uber* Trip,"[673e7106-fc5a-4af0-8149-94ad4819c309, 6738c8c...",5
64,Uiliannogueirades,"[6730f58c-0c98-4f9f-a1ef-02f77a6b3fad, 6731394...",2
65,Vamo Ki Vamo,[674c983d-2d23-4a81-9b5f-cd6f4dedbf7d],1
66,Vindi *Kinvo - Parcela 10/12,[65c1720a-7440-441b-88ff-0b14de27d2b8],1


In [24]:
df_grouped.explode('id_list')[['id_list', 'category']]

,id_list,category
0,6737e065-e913-4ef3-85e4-c3b624b73956,Teste
1,674dda0d-b620-4356-a00a-292ae7fbb188,Teste
1,674dda55-163b-4fe6-a87a-0e159df09a9e,Teste
1,675cc148-be8f-4e72-85e2-5fda484d765f,Teste
1,675f2e7a-9516-4ffe-8aab-3eef8c41c248,Teste
...,...,...
64,6730f58c-0c98-4f9f-a1ef-02f77a6b3fad,Teste
64,6731394a-9123-4291-9287-64e4e964ba93,Teste
65,674c983d-2d23-4a81-9b5f-cd6f4dedbf7d,Teste
66,65c1720a-7440-441b-88ff-0b14de27d2b8,Teste


In [ ]:
df_grouped['category'] = 'Teste'

df_raw = df_raw.merge(df_grouped.explode('id_list')[['id_list', 'category']], 
                      left_on='id', 
                      right_on='id_list', 
                      how='left').drop(columns=['id_list'])

,payee,type,date,user_date,amount,id,memo,sic,mcc,checknum,account_id,number,routing_number,institution,fid,year,year_month,in_out,is_installment,category
0,NaN,Credit Card,2024-12-04,NaN,-38.90,674ea97b-5319-4c30-83bf-47b17323cfe5,Cinemark Brasil,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Teste
1,NaN,Credit Card,2024-12-02,NaN,-14.96,674b9f46-f4c2-4ff9-98f9-722110c83e69,Uber Uber *Trip Help.U,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Teste
2,NaN,Credit Card,2024-12-02,NaN,-69.58,674c983d-2d23-4a81-9b5f-cd6f4dedbf7d,Vamo Ki Vamo,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Teste
3,NaN,Credit Card,2024-12-02,NaN,-7.65,674cb742-4e40-4dcb-a7db-98688abb227c,Top Sp Tarif*Descripti,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Teste
4,NaN,Credit Card,2024-12-02,NaN,-68.05,674c86d2-b258-43ce-b82e-45049ebba24a,Horti Fruti Ortencia,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Teste


In [69]:
df = pd.read_csv("D:/GitHub/_data/personal_finances/output/transactions_classified.csv", encoding='utf-8')
df.head()

,payee,type,date,user_date,amount,id,memo,sic,mcc,checknum,account_id,number,routing_number,institution,fid,year,year_month,in_out,is_installment,category
0,NaN,Credit Card,2024-12-04,NaN,-38.90,674ea97b-5319-4c30-83bf-47b17323cfe5,Cinemark Brasil,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Assinaturas e Serviços
1,NaN,Credit Card,2024-12-02,NaN,-14.96,674b9f46-f4c2-4ff9-98f9-722110c83e69,Uber Uber *Trip Help.U,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Transporte
2,NaN,Credit Card,2024-12-02,NaN,-69.58,674c983d-2d23-4a81-9b5f-cd6f4dedbf7d,Vamo Ki Vamo,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Compras
3,NaN,Credit Card,2024-12-02,NaN,-7.65,674cb742-4e40-4dcb-a7db-98688abb227c,Top Sp Tarif*Descripti,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Transporte
4,NaN,Credit Card,2024-12-02,NaN,-68.05,674c86d2-b258-43ce-b82e-45049ebba24a,Horti Fruti Ortencia,NaN,NaN,NaN,5a43cccd-bdaf-4211-a5c5-bc45326df942,5a43cccd-bdaf-4211-a5c5-bc45326df942,NaN,NU PAGAMENTOS S.A.,260,2024,2024-12,out,False,Horti Fruti


In [70]:
df['memo'].unique()

array(['Cinemark Brasil', 'Uber Uber *Trip Help.U', 'Vamo Ki Vamo',
       'Top Sp Tarif*Descripti', 'Horti Fruti Ortencia',
       'Boali Shopping Abc', 'Riachuelo 016 Grand Pl - Parcela 1/3',
       'Extra Hiper-1338', 'Wellhub Gympass Br Gym', 'Nagumo Santo Andre',
       'Cea Psa 125 Ecpc - Parcela 1/3', 'Br Souvenirs',
       'Bns Restaurante', 'Curipneus', 'Pg *Ton Mad Max Beer',
       'Cervejariarobeer', 'Playarte', 'Uber *Uber *Trip', 'Foodpork',
       'Gbr Barbearia', 'Ramalhao', 'Uber* Trip', 'Shopping Abc',
       'Moonlight Coffee Cafet', 'Netflix Entretenimento', 'Black Dogs',
       'Drogasil4188', 'Jessicadasilva',
       'C Pernambucanas Lj - Parcela 1/3', '1a99', 'Divinofogao',
       'Chiquinho Sorvetes', 'Restaurante Tasty Gril', 'Burger King',
       'Pani Di Grano', 'Facebk *Neeqbfymr2',
       'Mercadolivre*Bmf - Parcela 1/3', 'Openai *Chatgpt Subscr',
       'Mercadolivre*2produto', 'IOF de "Openai *Chatgpt Subscr"',
       'Mercadolivre*2produto - Parcela 1/3'

In [71]:
df['category'].value_counts()

category
Compras                          19
Transporte                       16
Compras Parceladas               13
Investimento                     13
Assinaturas e Serviços            9
Mercado                           6
Restaurantes                      6
Horti Fruti                       4
Transferências para terceiros     4
Receitas                          4
Moradia                           3
Fatura                            2
Telefone                          2
Saúde                             1
Name: count, dtype: int64

In [ ]:
df.query('category == "Compras Parceladas"')[['memo','category','amount']]

,memo,category,amount
6,Riachuelo 016 Grand Pl - Parcela 1/3,Compras Parceladas,-46.64
10,Cea Psa 125 Ecpc - Parcela 1/3,Compras Parceladas,-60.00
40,C Pernambucanas Lj - Parcela 1/3,Compras Parceladas,-65.34
50,Mercadolivre*Bmf - Parcela 1/3,Compras Parceladas,-19.85
57,Mercadolivre*2produto - Parcela 1/3,Compras Parceladas,-16.80
67,Cappta *St Vest - Parcela 2/4,Compras Parceladas,-79.87
68,Kalunga Shopping Abc - Parcela 3/3,Compras Parceladas,-49.96
69,Mlp *Netshoes - Parcela 4/4,Compras Parceladas,-92.49
70,Pg *Ton Gv Clear - Parcela 2/4,Compras Parceladas,-52.50
71,Riachuelo Lo*Riachuelo - Parcela 2/3,Compras Parceladas,-48.60
